# 01 — Huấn luyện baseline E0

Notebook này huấn luyện MobileNetV3-Small trên hai partition `train` và `validation` được cố định trong `results/split_manifest.csv`. Chỉ tập `train` của `tanganke/gtsrb` được đọc để huấn luyện.

Người dùng chạy thủ công từ trên xuống bằng kernel `robust-gtsrb-lite`. Không chạy notebook này bằng công cụ tự động.

## CẤU HÌNH

Các tham số thí nghiệm E0 được tập trung tại cell kế tiếp. Khi CUDA hết bộ nhớ, đổi `BATCH_SIZE` từ 64 xuống 32, restart kernel và chạy lại notebook từ đầu.

In [ ]:
import json
import os
import platform
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import load_dataset
from PIL import Image
from sklearn.metrics import precision_recall_fscore_support
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small

SEED = 42
IMAGE_SIZE = 128
NUM_CLASSES = 43
BATCH_SIZE = 64
NUM_WORKERS = 0
MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 3
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
EXPERIMENT_NAME = "E0_baseline"
DATASET_NAME = "tanganke/gtsrb"
ROOT = Path.cwd()
MANIFEST_PATH = ROOT / "results" / "split_manifest.csv"
CHECKPOINT_PATH = ROOT / "models" / "E0_baseline_best.pt"
RESULTS_DIR = ROOT / "results" / EXPERIMENT_NAME
DEVICE = torch.device("cuda")
USE_AMP = True


## Kiểm tra môi trường và khả năng tái lập

In [ ]:
def set_seed(seed):
    """Thiết lập seed cho các bộ sinh số ngẫu nhiên dùng trong notebook."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)


def check_environment():
    if platform.python_version() != "3.11.9":
        raise RuntimeError(f"Cần Python 3.11.9, đang dùng {platform.python_version()}")
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA không khả dụng. Dừng notebook; không tự chuyển sang CPU.")
    if torch.cuda.device_count() < 1:
        raise RuntimeError("Không tìm thấy GPU CUDA.")
    print(f"Python: {platform.python_version()}")
    print(f"PyTorch: {torch.__version__}")
    print(f"Torchvision: {__import__('torchvision').__version__}")
    print(f"CUDA runtime: {torch.version.cuda}")
    print(f"Thiết bị: {DEVICE}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")


set_seed(SEED)
check_environment()
torch.backends.cudnn.benchmark = False


## Đọc manifest và tạo DataLoader

In [ ]:
class IndexedImageDataset(Dataset):
    """Dataset wrapper chỉ đọc ảnh theo source index khi DataLoader yêu cầu."""

    def __init__(self, source_dataset, source_indices, image_transform):
        self.source_dataset = source_dataset
        self.source_indices = np.asarray(source_indices, dtype=np.int64)
        self.image_transform = image_transform

    def __len__(self):
        return len(self.source_indices)

    def __getitem__(self, item_index):
        source_index = int(self.source_indices[item_index])
        record = self.source_dataset[source_index]
        image = record["image"]
        if not isinstance(image, Image.Image):
            image = Image.fromarray(np.asarray(image))
        image = image.convert("RGB")
        label = int(record["label"])
        return self.image_transform(image), label


def build_dataloaders():
    if not MANIFEST_PATH.is_file():
        raise FileNotFoundError(f"Thiếu manifest: {MANIFEST_PATH}")
    manifest = pd.read_csv(MANIFEST_PATH)
    required_columns = {"source_index", "split"}
    if not required_columns.issubset(manifest.columns):
        raise ValueError("Manifest thiếu cột source_index hoặc split.")
    if set(manifest["split"].unique()) != {"train", "validation"}:
        raise ValueError("Manifest phải chỉ có hai partition train và validation.")
    if manifest["source_index"].duplicated().any():
        raise ValueError("Manifest có source_index trùng.")
    train_indices = manifest.loc[manifest["split"] == "train", "source_index"].astype(int).to_numpy()
    validation_indices = manifest.loc[manifest["split"] == "validation", "source_index"].astype(int).to_numpy()
    if len(train_indices) != 22644 or len(validation_indices) != 3996:
        raise ValueError(f"Kích thước partition sai: train={len(train_indices)}, validation={len(validation_indices)}")
    if len(set(train_indices).intersection(validation_indices)) != 0:
        raise ValueError("Train và validation bị overlap.")
    if len(train_indices) + len(validation_indices) != 26640:
        raise ValueError("Tổng partition không bằng 26640.")

    source_dataset = load_dataset(DATASET_NAME, split="train")
    if len(source_dataset) != 26640:
        raise ValueError(f"Kích thước tập train gốc sai: {len(source_dataset)}")
    source_labels = np.asarray(source_dataset["label"], dtype=np.int64)
    if source_labels.min() < 0 or source_labels.max() >= NUM_CLASSES:
        raise ValueError("Có label ngoài [0, 42].")
    partition_indices = np.concatenate([train_indices, validation_indices])
    partition_labels = source_labels[partition_indices]
    if len(np.unique(partition_labels)) != NUM_CLASSES:
        raise ValueError("Train/validation không đủ 43 lớp.")
    if sorted(partition_indices.tolist()) != list(range(len(source_dataset))):
        raise ValueError("Manifest không bao phủ đúng toàn bộ train gốc.")

    weights = MobileNet_V3_Small_Weights.DEFAULT
    weight_preprocessing = weights.transforms()
    normalization = {"mean": list(weight_preprocessing.mean), "std": list(weight_preprocessing.std)}
    train_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomRotation(10),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
        transforms.ToTensor(),
        transforms.Normalize(normalization["mean"], normalization["std"]),
    ])
    validation_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(normalization["mean"], normalization["std"]),
    ])
    train_dataset = IndexedImageDataset(source_dataset, train_indices, train_transform)
    validation_dataset = IndexedImageDataset(source_dataset, validation_indices, validation_transform)
    loader_generator = torch.Generator()
    loader_generator.manual_seed(SEED)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, generator=loader_generator)
    validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    label_feature = source_dataset.features["label"]
    class_names = list(label_feature.names) if hasattr(label_feature, "names") else [str(label) for label in range(NUM_CLASSES)]
    if len(class_names) != NUM_CLASSES:
        raise ValueError("Label mapping không có đúng 43 lớp.")
    return train_loader, validation_loader, class_names, normalization


train_loader, validation_loader, class_names, normalization = build_dataloaders()
print(f"Train: {len(train_loader.dataset)} mẫu; validation: {len(validation_loader.dataset)} mẫu")
print(f"Số lớp: {len(class_names)}")


## Xây dựng model và các hàm huấn luyện

In [ ]:
def build_model():
    weights = MobileNet_V3_Small_Weights.DEFAULT
    model = mobilenet_v3_small(weights=weights)
    input_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(input_features, NUM_CLASSES)
    return model.to(DEVICE)


def validate_finite(metrics):
    if not all(np.isfinite(float(value)) for value in metrics.values()):
        raise FloatingPointError(f"Loss hoặc metric không hữu hạn: {metrics}")


def train_one_epoch(model, data_loader, loss_function, optimizer, scaler):
    model.train()
    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    for images, labels in data_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(images)
            loss = loss_function(logits, labels)
        if not torch.isfinite(loss):
            raise FloatingPointError("Train loss là NaN hoặc Inf.")
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += float(loss.detach()) * labels.size(0)
        correct_predictions += int((logits.argmax(dim=1) == labels).sum())
        total_samples += labels.size(0)
    metrics = {"loss": total_loss / total_samples, "accuracy": correct_predictions / total_samples}
    validate_finite(metrics)
    return metrics


def evaluate_model(model, data_loader, loss_function):
    model.eval()
    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    all_predictions = []
    all_labels = []
    with torch.inference_mode():
        for images, labels in data_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(images)
                loss = loss_function(logits, labels)
            if not torch.isfinite(loss):
                raise FloatingPointError("Validation loss là NaN hoặc Inf.")
            predictions = logits.argmax(dim=1)
            total_loss += float(loss) * labels.size(0)
            correct_predictions += int((predictions == labels).sum())
            total_samples += labels.size(0)
            all_predictions.extend(predictions.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    precision, recall, macro_f1, _ = precision_recall_fscore_support(all_labels, all_predictions, average="macro", labels=list(range(NUM_CLASSES)), zero_division=0)
    metrics = {"loss": total_loss / total_samples, "accuracy": correct_predictions / total_samples, "macro_precision": precision, "macro_recall": recall, "macro_f1": macro_f1}
    validate_finite(metrics)
    return metrics


def save_checkpoint(model, optimizer, scheduler, epoch, best_metrics):
    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_epoch": epoch,
        "best_validation_macro_f1": best_metrics["macro_f1"],
        "best_validation_accuracy": best_metrics["accuracy"],
        "class_names": class_names,
        "label_mapping": {str(index): name for index, name in enumerate(class_names)},
        "model_name": "MobileNetV3-Small",
        "input_size": IMAGE_SIZE,
        "normalization": normalization,
        "experiment_config": {"seed": SEED, "batch_size": BATCH_SIZE, "max_epochs": MAX_EPOCHS, "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY, "early_stopping_patience": EARLY_STOPPING_PATIENCE, "experiment_name": EXPERIMENT_NAME},
        "seed": SEED,
    }
    temporary_path = CHECKPOINT_PATH.with_suffix(".tmp")
    torch.save(checkpoint, temporary_path)
    os.replace(temporary_path, CHECKPOINT_PATH)
    if not CHECKPOINT_PATH.is_file():
        raise IOError(f"Không tạo được checkpoint: {CHECKPOINT_PATH}")


def plot_training_history(history_frame):
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_frame["epoch"], history_frame["train_loss"], label="Train loss")
    axes[0].plot(history_frame["epoch"], history_frame["validation_loss"], label="Validation loss")
    axes[0].set_title("Loss theo epoch")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[1].plot(history_frame["epoch"], history_frame["train_accuracy"], label="Train accuracy")
    axes[1].plot(history_frame["epoch"], history_frame["validation_accuracy"], label="Validation accuracy")
    axes[1].plot(history_frame["epoch"], history_frame["validation_macro_f1"], label="Validation macro-F1")
    axes[1].set_title("Metric theo epoch")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    figure.tight_layout()
    figure.savefig(RESULTS_DIR / "training_curves.png", dpi=150)
    plt.close(figure)


## Huấn luyện E0 và lưu artifact

In [ ]:
model = build_model()
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
history_rows = []
best_epoch = None
best_metrics = None
best_macro_f1 = -float("inf")
epochs_without_improvement = 0
training_started_at = time.perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_started_at = time.perf_counter()
    try:
        train_metrics = train_one_epoch(model, train_loader, loss_function, optimizer, scaler)
        validation_metrics = evaluate_model(model, validation_loader, loss_function)
    except RuntimeError as error:
        if "out of memory" in str(error).lower():
            raise RuntimeError("CUDA out-of-memory. Hãy giảm BATCH_SIZE từ 64 xuống 32, restart kernel và chạy lại notebook từ đầu.") from error
        raise
    current_learning_rate = optimizer.param_groups[0]["lr"]
    epoch_seconds = time.perf_counter() - epoch_started_at
    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "validation_loss": validation_metrics["loss"],
        "validation_accuracy": validation_metrics["accuracy"],
        "validation_macro_precision": validation_metrics["macro_precision"],
        "validation_macro_recall": validation_metrics["macro_recall"],
        "validation_macro_f1": validation_metrics["macro_f1"],
        "learning_rate": current_learning_rate,
        "epoch_seconds": epoch_seconds,
    }
    history_rows.append(row)
    print(f"Epoch {epoch:02d}/{MAX_EPOCHS} | train loss={row['train_loss']:.4f} acc={row['train_accuracy']:.4f} | val loss={row['validation_loss']:.4f} acc={row['validation_accuracy']:.4f} macro-F1={row['validation_macro_f1']:.4f} | lr={current_learning_rate:.6g} | {epoch_seconds:.1f}s")
    if validation_metrics["macro_f1"] > best_macro_f1:
        best_macro_f1 = validation_metrics["macro_f1"]
        best_epoch = epoch
        best_metrics = validation_metrics.copy()
        save_checkpoint(model, optimizer, scheduler, best_epoch, best_metrics)
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
    scheduler.step()
    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Dừng sớm sau epoch {epoch} theo validation macro-F1.")
        break

total_training_seconds = time.perf_counter() - training_started_at
if best_epoch is None or best_metrics is None or not CHECKPOINT_PATH.is_file():
    raise RuntimeError("Không có best checkpoint hợp lệ sau training.")
history_frame = pd.DataFrame(history_rows)
if history_frame.empty or len(history_frame) != len(history_rows):
    raise RuntimeError("Không tạo được history hợp lệ.")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
config = {
    "experiment_name": EXPERIMENT_NAME,
    "dataset_name": DATASET_NAME,
    "seed": SEED,
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "torchvision_version": __import__('torchvision').__version__,
    "device": str(DEVICE),
    "gpu_name": torch.cuda.get_device_name(0),
    "cuda_runtime": torch.version.cuda,
    "model_name": "MobileNetV3-Small",
    "image_size": IMAGE_SIZE,
    "num_classes": NUM_CLASSES,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "max_epochs": MAX_EPOCHS,
    "epochs_ran": len(history_rows),
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "optimizer": "AdamW",
    "scheduler": "CosineAnnealingLR",
    "loss": "CrossEntropyLoss",
    "automatic_mixed_precision": USE_AMP,
    "normalization": normalization,
    "train_samples": len(train_loader.dataset),
    "validation_samples": len(validation_loader.dataset),
}
(RESULTS_DIR / "config.json").write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
history_frame.to_csv(RESULTS_DIR / "history.csv", index=False, encoding="utf-8")
validation_metrics_artifact = {"best_epoch": best_epoch, **best_metrics}
(RESULTS_DIR / "validation_metrics.json").write_text(json.dumps(validation_metrics_artifact, ensure_ascii=False, indent=2), encoding="utf-8")
plot_training_history(history_frame)
print(f"Đã lưu checkpoint tốt nhất tại {CHECKPOINT_PATH}")
print(f"Đã lưu artifact tại {RESULTS_DIR}")


## TÓM TẮT TRẠNG THÁI

In [ ]:
required_artifacts = [
    CHECKPOINT_PATH,
    RESULTS_DIR / "config.json",
    RESULTS_DIR / "history.csv",
    RESULTS_DIR / "validation_metrics.json",
    RESULTS_DIR / "training_curves.png",
]
missing_artifacts = [str(path) for path in required_artifacts if not path.is_file()]
if missing_artifacts:
    raise RuntimeError(f"ERROR: thiếu artifact: {missing_artifacts}")
print("SUCCESS")
print(f"Số epoch đã chạy: {len(history_rows)}")
print(f"Best epoch: {best_epoch}")
print(f"Best validation accuracy: {best_metrics['accuracy']:.6f}")
print(f"Best validation macro-F1: {best_metrics['macro_f1']:.6f}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print("Artifact:")
for artifact_path in required_artifacts:
    print(f"- {artifact_path}")
print(f"Tổng thời gian training: {total_training_seconds:.2f} giây")
print(f"GPU: {torch.cuda.get_device_name(0)}")
